# Milestone 4 — M4B: ray-to-volume deposition

Exact **ray–tet** deposition → volumetric source `S_clean` with energy conservation.

Plan: [`plans/milestone_04/04_diffusion_plan.md`](../../plans/milestone_04/04_diffusion_plan.md). Previous: `04a`. Next: `04c`.

## Checks

1. Energy conservation: `sum(S_clean * volume) ≈ total_scattered`.
2. `S_clean ≥ 0`; zero when `μ_s = 0`.
3. Monotonic profiles along a known axis ray (no phantom centroid deposits).


In [ ]:
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

!pip install --quiet --no-cache-dir "{ROOT}[fem]" -c "{ROOT}/requirements.txt"

from gummybear.paths import display_path

print(f"ROOT={{display_path(ROOT)}}")



In [2]:
from gummybear.geometry import inspect_stl
from gummybear.optics import deposit_ray_source, generate_diffusion_mesh
from gummybear_validation.helpers import (
    assert_deposition_conservation,
    assert_deposition_sanity,
    make_centroid_axis_ray,
    print_deposition_summary,
)
from gummybear_validation.plotting import (
    plot_deposition_scene,
    plot_profile_along_ray,
)


In [3]:
inspection = inspect_stl(ROOT / "cad" / "proto_bear_head.stl")
mesh = inspection["mesh"]
diff_mesh = generate_diffusion_mesh(mesh, target_elements=2000)
volumes = diff_mesh.volumes


In [ ]:
ray, p0, p1 = make_centroid_axis_ray(diff_mesh, axis="x", intensity=1.0)

material = dict(mu_s=0.2, mu_a=0.1)
result = deposit_ray_source(diff_mesh, ray, **material)

print_deposition_summary(result)
assert_deposition_sanity(result, diff_mesh.n_tets)
assert_deposition_conservation(result, volumes)


In [ ]:
zero_scatter = deposit_ray_source(diff_mesh, ray, mu_s=0.0, mu_a=1.0)
assert zero_scatter.total_scattered == 0.0
assert zero_scatter.S_clean.max() == 0.0
print("μ_s=0 ⇒ no scattering source (expected).")


In [ ]:
plot_deposition_scene(diff_mesh.centroids, result.E_scat_elem, p0, p1)
plot_profile_along_ray(
    diff_mesh.centroids,
    result.E_scat_elem,
    p0,
    p1,
    ylabel="E_scat_elem",
    title="Element-integrated scattered energy along ray",
)
plot_profile_along_ray(
    diff_mesh.centroids,
    result.S_clean,
    p0,
    p1,
    ylabel="S_clean",
    title="Volumetric source density along ray",
)
